## Hypothesis Testing

In [27]:
import pandas as pd
import scipy.stats as stats

def get_season(month):
    if month in [12, 1, 2]: return 'Winter'
    elif month in [3, 4, 5]: return 'Spring'
    elif month in [6, 7, 8]: return 'Summer'
    else: return 'Autumn'

H0 = Discord messages in 2023 are evenly distributed  
HA = Discord messages in 2023 are **not** evenly distributed

Test: Chi-Square test of Goodness of Fit (GOF)

In [28]:
df = pd.read_csv('data/anonymized/discord/messages.csv')

# Convert date column to datetime objects
df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d %H:%M:%S')

# Filter exclusively for 2023
df = df[(df['date'].dt.year == 2023)]

df['season'] = df['date'].dt.month.apply(get_season)

print("\n--- Chi-Square Goodness-of-Fit (Total Volume) ---")
season_totals = df['season'].value_counts()
print(f"Total Messages by Season:\n{season_totals.to_string()}")

# Run Chi-Square
chi2_stat, chi2_p = stats.chisquare(f_obs=season_totals.values)

print(f"Chi-Square P-Value: {chi2_p:.4e}")
if chi2_p < 0.075:
    print("Result: Reject H0. The total messages are NOT uniformly distributed.")
else:
    print("Result: Fail to reject H0. The total messages are evenly distributed.")


--- Chi-Square Goodness-of-Fit (Total Volume) ---
Total Messages by Season:
season
Summer    804
Autumn    475
Winter    163
Spring    101
Chi-Square P-Value: 6.6959e-176
Result: Reject H0. The total messages are NOT uniformly distributed.


H0 = The average daily message count is the same across all seasons in 2023.  
HA = The average daily message count is higher in Summer 2023 compared to other seasons in 2023.

Test: ANOVA (Analysis of Variance)

In [29]:
df = pd.read_csv('data/anonymized/discord/messages.csv')

# Convert date column to datetime objects
df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d %H:%M:%S')

# Filter exclusively for 2023
df = df[(df['date'].dt.year == 2023)]

df['season'] = df['date'].dt.month.apply(get_season)

print("\n--- One-Way ANOVA (Daily Averages) ---")
# Extract just the calendar day (ignoring time) to count daily messages
df['just_date'] = df['date'].dt.date
daily_counts = df.groupby(['just_date', 'season']).size().reset_index(name='msg_count')

# Separate the daily counts into lists by season
summer = daily_counts[daily_counts['season'] == 'Summer']['msg_count']
winter = daily_counts[daily_counts['season'] == 'Winter']['msg_count']
spring = daily_counts[daily_counts['season'] == 'Spring']['msg_count']
autumn = daily_counts[daily_counts['season'] == 'Autumn']['msg_count']

print(f"Average Daily Messages - Summer: {summer.mean():.1f}")
print(f"Average Daily Messages - Winter: {winter.mean():.1f}")
print(f"Average Daily Messages - Spring: {spring.mean():.1f}")
print(f"Average Daily Messages - Autumn: {autumn.mean():.1f}")

# Run ANOVA
anova_stat, anova_p = stats.f_oneway(summer, winter, spring, autumn)

print(f"ANOVA P-Value: {anova_p:.4e}")
if anova_p < 0.075:
    print("Result: Reject H0. The daily averages vary significantly by season.")
else:
    print("Result: Fail to reject H0. No significant difference in daily averages.")


--- One-Way ANOVA (Daily Averages) ---
Average Daily Messages - Summer: 12.4
Average Daily Messages - Winter: 7.8
Average Daily Messages - Spring: 4.8
Average Daily Messages - Autumn: 9.1
ANOVA P-Value: 5.7661e-02
Result: Reject H0. The daily averages vary significantly by season.


Final conclusion based on the 2 tests: Was I more active on Discord in Summer 2023 compared to other seasons in 2023?

In [30]:
if (anova_p < 0.075) and (summer.mean() > winter.mean()) and (summer.mean() > spring.mean()) and (summer.mean() > autumn.mean()):
    print("\nData is not uniform, and Summer is statistically the most active season.")
elif anova_p >= 0.075 or chi2_p >= 0.075:
    print("\nMessage distribution is uniform.")
else:
    print("\nData is not uniform, but Summer is not definitively the highest peak.")


Data is not uniform, and Summer is statistically the most active season.
